# 🌿 Plant Disease Detection — CNN Training
**Madda Walabu University | Morketa Negash (Ugrr/51983/15)**

### Steps:
1. Upload your dataset ZIP
2. Run all cells top to bottom
3. Download the trained model at the end
4. Put it in your `model/` folder

> ⚡ Make sure GPU is enabled: **Runtime → Change runtime type → T4 GPU**

## ✅ Step 1 — Check GPU

In [ ]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))
print('')
if tf.config.list_physical_devices('GPU'):
    print('✅ GPU is ready — training will be FAST!')
else:
    print('❌ No GPU detected — go to Runtime → Change runtime type → T4 GPU')

## ✅ Step 2 — Upload your dataset ZIP file

You need to ZIP your `dataset` folder first:
- On Windows: right-click the `dataset` folder → Send to → Compressed (zipped) folder
- Name it: `dataset.zip`
- Then run this cell and upload it

In [ ]:
from google.colab import files
import zipfile, os

print('📁 Please upload your dataset.zip file...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'\n📦 Extracting {zip_name}...')
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/dataset_raw')

# Find the actual dataset folder
import glob
for root, dirs, files_list in os.walk('/content/dataset_raw'):
    subdirs = [d for d in dirs if not d.startswith('.')]
    if len(subdirs) >= 5:
        DATASET_DIR = root
        break

print(f'\n✅ Dataset found at: {DATASET_DIR}')
classes = sorted(os.listdir(DATASET_DIR))
total = 0
for c in classes:
    p = os.path.join(DATASET_DIR, c)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        total += n
        print(f'  {c}: {n} images')
print(f'\n📊 TOTAL: {len(classes)} classes, {total} images')

## ✅ Step 3 — Build CNN Model

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import numpy as np
import json, os

# ── Config ──
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 0.001

# ── Data generators with augmentation ──
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

NUM_CLASSES = len(train_gen.class_indices)
print(f'✅ Classes: {NUM_CLASSES}')
print(f'✅ Training images: {train_gen.samples}')
print(f'✅ Validation images: {val_gen.samples}')

# ── Save class names ──
class_indices = {v: k for k, v in train_gen.class_indices.items()}
with open('/content/class_names.json', 'w') as f:
    json.dump(class_indices, f, indent=2)
print(f'\n✅ Class names saved:')
for k, v in sorted(class_indices.items()):
    print(f'  {k}: {v}')

In [ ]:
# ── Build CNN Architecture ──
def build_model(num_classes):
    model = keras.Sequential([
        keras.Input(shape=(224, 224, 3)),

        # Block 1
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Block 4
        layers.Conv2D(256, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(256, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Block 5
        layers.Conv2D(512, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Fully connected
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),

        # Output
        layers.Dense(num_classes, activation='softmax'),
    ])
    return model

model = build_model(NUM_CLASSES)

optimizer = keras.optimizers.SGD(learning_rate=LR, momentum=0.9, nesterov=True)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## ✅ Step 4 — Train the Model

In [ ]:
# ── Callbacks ──
checkpoint = ModelCheckpoint(
    '/content/plant_disease_cnn.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# ── Train ──
print('🚀 Starting training...')
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=[checkpoint, early_stop, reduce_lr],
    verbose=1
)

print('\n✅ Training complete!')

## ✅ Step 5 — Evaluate & Plot Results

In [ ]:
import matplotlib.pyplot as plt

# Final accuracy
val_loss, val_acc = model.evaluate(val_gen, verbose=0)
print(f'\n=========================================')
print(f'  FINAL RESULTS')
print(f'=========================================')
print(f'  Validation Accuracy : {val_acc*100:.2f}%')
print(f'  Validation Loss     : {val_loss:.4f}')
print(f'=========================================')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Plant Disease CNN — Training Results', fontsize=14)

axes[0].plot(history.history['accuracy'], label='Train', color='blue')
axes[0].plot(history.history['val_accuracy'], label='Validation', color='orange')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train', color='blue')
axes[1].plot(history.history['val_loss'], label='Validation', color='orange')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=150)
plt.show()
print('✅ Training plot saved!')

## ✅ Step 6 — Download Trained Model Files

Download both files and put them in your project's `model/` folder.

In [ ]:
from google.colab import files

print('📥 Downloading plant_disease_cnn.h5 ...')
files.download('/content/plant_disease_cnn.h5')

print('📥 Downloading class_names.json ...')
files.download('/content/class_names.json')

print('📥 Downloading training_history.png ...')
files.download('/content/training_history.png')

print('')
print('✅ Done! Now:')
print('  1. Copy plant_disease_cnn.h5  →  model/plant_disease_cnn.h5')
print('  2. Copy class_names.json      →  model/class_names.json')
print('  3. Run: streamlit run app.py')